# Make Predictions on Training Data

In [1]:
# Import Packages
import ultralytics
from ultralytics import YOLO
import numpy as np
import time
import yaml
from pathlib import Path
import torch
import pandas as pd
from shutil import copy2

#-- Directories and Paths --#
REPO_ROOT = Path("..")      

DATA_CONFIG_PATH = REPO_ROOT / 'configurations' / 'model_data-seg.yaml'

#-- Model --#

# Load model parameters / overrides
with open(REPO_ROOT / 'configurations' / 'val_model_overrides.yaml', 'r') as f:
    overrides = yaml.safe_load(f)

# Override path in your dictionary
overrides['data'] = str(DATA_CONFIG_PATH)

# Assign Trained Model Weights
def resolve(p: str | Path) -> Path:
    p = Path(p)
    return p if p.is_absolute() else (REPO_ROOT / p)

weights_path = resolve(overrides.pop("weights"))

# Load Trained Model's Weights
model = YOLO(str(weights_path))  

# Predictions from the model
with torch.inference_mode():
    metrics = model.val(**overrides)


#-- Results --#

#--- Results Directory ---#
# where YOLO saved this run
out_dir = Path(metrics.save_dir)
out_dir.mkdir(parents=True, exist_ok=True)

#--- Precision & Recall Calculations ---#

# names -> {id: name}; confusion matrix includes a last row/col for "background"
names = [metrics.names[i] for i in range(len(metrics.names))]
cm = metrics.confusion_matrix.matrix.astype(np.int64)   # shape = (nc+1, nc+1) # cm = confusion_matrix
nc = len(names) # nc = name classes

# diagonal (exclude background) = TP
tp = np.diag(cm)[:nc]

# row sums (include background col) minus TP = FP for that predicted class
fp = cm[:nc, :].sum(axis=1) - tp

# column sums (include background row) minus TP = FN for that true class
fn = cm[:, :nc].sum(axis=0) - tp

precision = tp / (tp + fp + 1e-16)
recall    = tp / (tp + fn + 1e-16)
f1        = 2 * precision * recall / (precision + recall + 1e-16)

df = pd.DataFrame({
    "class": names, "TP": tp, "FP": fp, "FN": fn,
    "precision": precision, "recall": recall, "f1": f1
})
print(df)
# Output P&R Results
df.to_csv(out_dir / "P&R_results.csv", header=True,index=True)   
print(f"Saved df results metrics to {out_dir/'df_results.csv'}")


#-- Export Validation Results --#
# dataframe
validation_df = metrics.to_df()

# validation_df.to_csv(csv_path, header=True, index=True)
summary_df = pd.DataFrame([metrics.results_dict])  # <-- dict -> one-row DataFrame
summary_df.to_csv(out_dir / "summary_metrics_results.csv", index=False)
print(f"Saved metrics to {out_dir/'results.csv'}")




# choose task: 'seg' for masks, 'box' for boxes
task = 'seg' # Choosing specific 'task' type of 'segmentation' selects for metric associates
seg_metrics = getattr(metrics, task)

print(f"AP@0.75 ({task}): {seg_metrics.map75:.4f}")
print(f"AP@0.50 ({task}): {seg_metrics.map50:.4f}")
print(f"Classes  mAP@0.50:0.95 ({task}): {seg_metrics.map:.4f}")

# Per-class mAP (COCO mAP@0.50:0.95) if exposed
names = metrics.names if isinstance(metrics.names, (list, tuple)) else list(metrics.names.values())
if hasattr(seg_metrics, "maps") and seg_metrics.maps is not None:
    for i, name in enumerate(names):
        print(f"{name:>16}  mAP@0.50:0.95 = {seg_metrics.maps[i]:.4f}")


# Copy test_parameters into output folder
test_model_overrides = Path(REPO_ROOT / 'configurations/val_model_overrides.yaml')
copy2(test_model_overrides, metrics.save_dir / test_model_overrides.name )
try:
    print(f'"val_model_parameters" Copied to : {metrics.save_dir}')
except Exception:
    print('Error - Parameters Not Copied To Output')

# State output location
try:
    print("Output Metrics Saved to:", metrics.save_dir)
except Exception:
    print('Error - Location Not Printed')

Ultralytics 8.3.185  Python-3.11.14 torch-2.5.1 CPU (Intel Xeon Gold 6230R CPU @ 2.10GHz)
YOLO11s-seg summary (fused): 113 layers, 10,067,590 parameters, 0 gradients, 32.8 GFLOPs
val: Fast image access  (ping: 0.00.0 ms, read: 1593.482.8 MB/s, size: 3072.3 KB)


val: Scanning C:\Users\hmanasi1\Documents\ADML\solafune_tree_canopy\data\processed\labels\val.cache... 30 images, 0 bac
                 Class     Images  Instances      Box(P          R      mAP50  mAP50-95)     Mask(P          R      mAP

requirements: Ultralytics requirement ['faster-coco-eval>=1.6.7'] not found, attempting AutoUpdate...

requirements: AutoUpdate success  6.0s
WARNING requirements: Restart runtime or rerun command for updates to take effect

WARNING Limiting validation plots to first 50 items per image for speed...


                 Class     Images  Instances      Box(P          R      mAP50  mAP50-95)     Mask(P          R      mAP

WARNING Limiting validation plots to first 50 items per image for speed...


                 Class     Images  Instances      Box(P          R      mAP50  mAP50-95)     Mask(P          R      mAP

WARNING Limiting validation plots to first 50 items per image for speed...


                 Class     Images  Instances      Box(P          R      mAP50  mAP50-95)     Mask(P          R      mAP


                   all         30       7960       0.64      0.207      0.429      0.248       0.67      0.209      0.443      0.218
       individual_tree         30       7104      0.695      0.347       0.53      0.302      0.684      0.341      0.521       0.25
        group_of_trees         26        856      0.586     0.0678      0.329      0.193      0.657     0.0759      0.365      0.185
Speed: 9.0ms preprocess, 243.8ms inference, 0.0ms loss, 430.3ms postprocess per image
Saving runs\segment\val_Yolo11s_canopy_832_adamW_5\predictions.json...
Results saved to runs\segment\val_Yolo11s_canopy_832_adamW_5
             class    TP   FP    FN  precision    recall        f1
0  individual_tree  2550  998  4554   0.718715  0.358953  0.478783
1   group_of_trees    64   35   792   0.646465  0.074766  0.134031
Saved df results metrics to runs\segment\val_Yolo11s_canopy_832_adamW_5\df_results.csv
Saved metrics to runs\segment\val_Yolo11s_canopy_832_adamW_5\results.csv
AP@0.75 (seg): 0.1936


# Make Prediction On Evaluation Data

In [4]:
import ultralytics
from ultralytics import YOLO
import numpy as np
import time
import yaml
from pathlib import Path
import torch
from datetime import datetime

current_dt = datetime.now().strftime('%Y-%m-%d')
REPO_ROOT = Path("..")   

# Load model parameters / overrides YAML
with open(REPO_ROOT / 'configurations' / 'predict_model_overrides.yaml', 'r') as f: # Modify overrides where necessary
    overrides = yaml.safe_load(f)

# Make YAML Paths Absolute
for key in ("source", "project"):
    if key in overrides:
        overrides[key] = str((REPO_ROOT / overrides[key]).resolve())

# Load Trained Model Weights
def resolve(p: str | Path) -> Path:
    p = Path(p)
    return p if p.is_absolute() else (REPO_ROOT / p)

weights_path = resolve(overrides.pop("weights"))

# Model Naming
# optional: use the training run name from the weights path
train_run = weights_path.parents[1].name  # e.g., train_Yolov8s_canopy_832_...
run_name = f"predict_{train_run}_{current_dt}"

# inject name (and project if you want a different subfolder)
overrides.setdefault("project", str(REPO_ROOT / "runs"))
overrides["name"] = run_name
# overrides["exist_ok"] = True  # set True to overwrite instead of auto-appending _2

# Load Trained Model's Weights
model = YOLO(str(weights_path))  

# Predictions from the model
with torch.inference_mode():
    predictions = model.predict(**overrides)

# State output location

# Stream = True
for last in predictions:
    pass
print("Saved to:", last.save_dir)

# Stream = False
# print("Outputs saved to:", predictions[0].save_dir)


image 1/150 C:\Users\hmanasi1\Documents\ADML\solafune_tree_canopy\data\processed\images\predict\10cm_evaluation_1.tif: 1024x1024 53 individual_trees, 344.3ms
image 2/150 C:\Users\hmanasi1\Documents\ADML\solafune_tree_canopy\data\processed\images\predict\10cm_evaluation_10.tif: 1024x1024 63 individual_trees, 3 group_of_treess, 236.4ms
image 3/150 C:\Users\hmanasi1\Documents\ADML\solafune_tree_canopy\data\processed\images\predict\10cm_evaluation_11.tif: 1024x1024 229 individual_trees, 3 group_of_treess, 261.2ms
image 4/150 C:\Users\hmanasi1\Documents\ADML\solafune_tree_canopy\data\processed\images\predict\10cm_evaluation_12.tif: 1024x1024 77 individual_trees, 7 group_of_treess, 249.8ms
image 5/150 C:\Users\hmanasi1\Documents\ADML\solafune_tree_canopy\data\processed\images\predict\10cm_evaluation_13.tif: 1024x1024 165 individual_trees, 335.4ms
image 6/150 C:\Users\hmanasi1\Documents\ADML\solafune_tree_canopy\data\processed\images\predict\10cm_evaluation_14.tif: 1024x1024 90 individual_tr

In [12]:
# Packages
from pathlib import Path
import pandas as pd
import numpy as np
import json
from PIL import Image
import pandas as pd
from datetime import datetime

# Create paths
REPO_ROOT = Path.cwd()

CURRENT_DATETIME = datetime.now().strftime('%Y-%m-%d')

# Prediction Annotations
labels = "../runs/segment/predict_train_Yolo11s_canopy_832_adamW__2025-11-02\labels" # Modify Prediction Txt annotations where necessary
IMGS = '../data/processed/images/predict'
OUT_JSON = REPO_ROOT / 'exports'/ str(CURRENT_DATETIME) / 'submission.json'



#---Define Functions---#

# Image Metadata
def get_meta(IMGS):

    meta_data = {}
    scene_type_default = "Unknown"
    IMGS = Path(IMGS)

    if IMGS.exists():

        for p in Path(IMGS).iterdir():
            if p.suffix.lower() not in {".tif"}: # All images .tif
                continue
            file_name = p.name
            cm_resolution = str(file_name[0:2])

            with Image.open(p) as image: # Pillow / Pill package
                width, height = image.size

            meta_data[file_name] = {
                'file_name' : file_name,
                "width" : width,
                'height' : height,
                "cm_resolution" : cm_resolution,
                "scene_type" : scene_type_default

            }
    else:
      print('Error - IMGs Not Exist')
    return meta_data

# Get Annotations
def get_annotations(LABELS):
    annotations = []

    for txt_file in sorted(Path(LABELS).iterdir()):
        
        if txt_file.is_file() and txt_file.suffix.lower() == ".txt":
            image_name = txt_file.name

            with open(txt_file, 'r') as file:
                for line in file:

                    # 1. Split the line into tokens
                    tokens = line.strip().split()

                    # 2. Convert tokens into usable pieces
                    class_id = int(tokens[0])                    # first token
                    *coords, confidence_level = map(float, tokens[1:])  
                    # all middle tokens are coords, last one is confidence
                    

                    # 3. Group coords into pairs (x,y)
                    segmentation = [
                        (coords[i], coords[i+1])
                        for i in range(0, len(coords), 2)
                    ]

                    # 4. Save everything into dict
                    annotations.append({
                        "image": image_name,
                        "class": class_id,
                        "confidence_level": confidence_level,
                        "segmentation": segmentation
                    })

    return annotations


# Denormalise segmentations to pixels

def denorm_segmentation(seg, width, height, *, flatten=True, round_int=True):
    """
    seg: list of (x,y) pairs in [0,1]
    returns: flat pixel list [x1,y1,x2,y2,...] (default) or list of pairs
    """
    # If seg is already flat [x1,y1,...], turn into pairs
    if seg and not isinstance(seg[0], (list, tuple)):
        it = iter(seg)
        seg_pairs = list(zip(it, it))
    else:
        seg_pairs = seg

    out = []
    for x, y in seg_pairs:
        X = max(0.0, min(width  - 1, x * width))
        Y = max(0.0, min(height - 1, y * height))
        if round_int:
            X = int(round(X))
            Y = int(round(Y))
        if flatten:
            out.extend([X, Y])
        else:
            out.append((X, Y))
    return out


# Append annotations to images

def append_imgs_annotations(img_list, annot_list):
    # ensure each image has an annotations list
    for img in img_list:
        if "annotations" not in img or img["annotations"] is None:
            img["annotations"] = []

    for image in img_list:
        w, h = image["width"], image["height"]
        fname = image["file_name"]

        # all annots for this image
        matched = (a for a in annot_list if a["image"] == fname)

        for a in matched:
            seg_px = denorm_segmentation(a["segmentation"], w, h, flatten=True, round_int=True) # Denormalise Segemetation to Pixels

            image["annotations"].append({
                "class": a["class"],
                "confidence_score": float(a["confidence_level"]),  # or keep as formatted string if you prefer
                "segmentation": seg_px
            })

    return img_list


# class id
ID_to_Name = {
    0 : 'individual_tree',
    1 : 'group_of_trees'
}



#--Procedure--#
print('Get Meta Data')
meta = get_meta(IMGS) # Retrieve Meta Data
meta_2items = list(meta.items())[:1]
meta_2keys = list(meta)[:1]

print(f'Meta Data Items : {meta_2items}')

print(f'Meta Data Keys : {meta_2items}')


images = [] # Produce Images list

print('Append Meta Data To Images List')
for m in meta.values(): # Append Meta Data to Images List
    images.append({
        'file_name' : m['file_name'],
        'width' : m['width'],
        'height' : m['height'],
        'cm_resolution' : m['cm_resolution'],
        'scene_type' : m['scene_type'],
        'annotations' : []
    })
    

images_sorted = sorted(images, key=lambda x: x['file_name']) # Sort Images
print(f'images_sorted ')
    
print('Get Annotations')
annotations_list = get_annotations(labels) # Retrieve Annotations & Denormalise Segmentations to Pixels

for annot in annotations_list: # Clean Annotations
    # .txt suffix -> .tif
    annot['image'] = annot['image'][:-4] + '.tif'

    # map class id -> name
    annot['class'] = ID_to_Name.get(annot['class'], 'Unknown')

    # convert confidence_level -> percentage string
    annot['confidence_level'] = f"{annot['confidence_level']:.2f}"

if annotations_list and len(annotations_list[0]) > 0:
    print('Annotations List Completed')

appended_list = append_imgs_annotations(images_sorted, annotations_list) # Append Annotations to Images

print(f'Appended_list Outputed ')

predict_answer = {}
predict_answer = {'images' : appended_list}
print(f'Predict_Answer Formulated')


# Assign Scene Type from Sample Answer to Submission

sample_answer_input = REPO_ROOT / 'exports/sample_answer.json' # Acquire Sample Answer path 

with open(sample_answer_input) as f: # Open Sample Answer Structure
    sample_answer = json.load(f)

sample_answer = sample_answer['images']


image_scenes = [] # Create list of images and associated 'scene_type'

for image in sample_answer:
    file_name = image.get('file_name') # GET each image's name
    scene_type = image.get('scene_type') # GET each image's scene_type

    image_scenes.append(  # Append to list
        {
            'filename':file_name, 
            'scene_type':scene_type
        }
    )


pd.json_normalize(image_scenes) #-- Potential redunant code --#

image_scenes_df = pd.DataFrame(image_scenes) # Transform Image_scenes list into DF

scene_map = image_scenes_df.set_index('filename')['scene_type'].to_dict() # Restructure image_scenes DF

submission = predict_answer

print('Append scene_types to predict_answer')
# Update each image in submission['images']
for image in submission['images']:
    filename = image['file_name']
    if image['scene_type'] == "Unknown" and filename in scene_map:
        image['scene_type'] = scene_map[filename]



print('Export Submission')
# Rewrite Submission JSON File
OUT_JSON.parent.mkdir(parents=True, exist_ok=True)
with open(OUT_JSON, "w") as f:
    json.dump(submission, f, indent=2)


print(f"Export Submission Complete \
      Saved As: 'Submission' \
      Saved At : {OUT_JSON}")


Get Meta Data
Meta Data Items : [('10cm_evaluation_1.tif', {'file_name': '10cm_evaluation_1.tif', 'width': 1024, 'height': 1024, 'cm_resolution': '10', 'scene_type': 'Unknown'})]
Meta Data Keys : [('10cm_evaluation_1.tif', {'file_name': '10cm_evaluation_1.tif', 'width': 1024, 'height': 1024, 'cm_resolution': '10', 'scene_type': 'Unknown'})]
Append Meta Data To Images List
images_sorted 
Get Annotations
Annotations List Completed
Appended_list Outputed 
Predict_Answer Formulated
Append scene_types to predict_answer
Export Submission
Export Submission Complete       Saved As: 'Submission'       Saved At : C:\Users\hmanasi1\Documents\ADML\solafune_tree_canopy\notebooks\exports\2025-11-02\submission.json
